In [1]:
import json
import os
import pandas as pd
import torch
import numpy as np

from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from threading import Thread
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from util import *
from prompts.synonym_context_prompt import *

load_dotenv()

pd.set_option('display.max_rows', None)
#"gpt-4.1-mini"
#"gpt-4o-mini"
#"gpt-4.1"
#"gpt-4o"

model_ = "gpt-5.1"
api_key = os.getenv("API_KEY")


llm = llm_call(model_version = model_, api_key= api_key)

In [2]:
# "Credit-TEST-POLLUTED.NORND-activity-0.3-0"
# "Credit-TEST-SYNONYM-0.3-0"
#Credit-TRAIN-DISTORTED-activity-0.3-0
#Pub-Collateral
#Credit-TRAIN-HOMONYM-0.3-0
#Credit-TEST-CLEAN
test = "Credit-TEST-SYNONYM-0.3-0"
LOG_NAME = f"./dataset/{test}.csv" 

df_new, cases_json = build_event_jsons(log_name = LOG_NAME, chunk_cases = 10)

#sample_size = int(len(df_new['case_id'].unique()) * 0.25)
#np.random.seed(42)
#selected_case_ids = np.random.choice(df_new['case_id'].unique(), size=sample_size, replace=False)
#df_new = df_new[df_new['case_id'].isin(selected_case_ids)].copy()


df_new.head(3)


,event_id,case_id,activity,timestamp,label
0,0,15,Check for completeness,2023-09-29 16:58:01.944,NaN
1,1,15,New online application received,2023-09-29 16:58:01.944,NaN
2,2,15,check application,2023-09-29 17:14:52.875,synonymous Label(Activity)


In [3]:
activity_list = df_new['activity'].unique().tolist()
activity_list_json = json.dumps(activity_list, indent=4, ensure_ascii=False)
print(activity_list_json)

[
    "Check for completeness",
    "New online application received",
    "check application",
    "Make decision",
    "send notification reject",
    "time out",
    "EVENT 13 END",
    "check no missing",
    "application received",
    "Request info",
    "information received",
    "assess completeness",
    "Perform checks",
    "review request received",
    "verify completeness",
    "info received",
    "Notify accept",
    "Deliver card",
    "require additional details",
    "notify reject",
    "execute checks",
    "deliver package",
    "send card",
    "get new application",
    "obtain information",
    "check complete",
    "request more detail",
    "decide",
    "send notification accept",
    "request more information",
    "send package",
    "take decision",
    "revision received"
]


In [4]:
SYSTEM_PROMPT_STEP1 = """
You are an expert Process Mining Data Pre-processor.
Your goal is to filter a raw list of activity names based on specific criteria provided in the User Prompt.

### KNOWLEDGE BASE: IMPERFECTION PATTERNS
Use these definitions to identify which labels belong to which category.

1.  **Polluted Labels (Mutable Qualifiers):**
    Labels that share a immutable boiler-plate text but differ due to mutable text (e.g., embedded IDs or codes).
    * **Detection Criteria:**
        * **Long Numeric IDs:** 8+ digits (e.g., `20260122`, `9988776655`).
        * **Mixed Codes:** 6+ alphanumeric chars (e.g., `XJ9281`, `Ref_A1B2C3`).
        * **Delimiters:** Attached via `_`, `-`, `:`, `/`, `#`, `.`, or space.

2.  **Distorted Labels (Character-Level Corruption):**
    Labels containing specific character-level corruptions (typos, OCR faults) of a canonical form. Unlike synonyms, these are "Noise".
    * **Detection Criteria:**
        1.  **Case Mutation:** Identical spelling, different capitalization (e.g., "Open" vs "open" vs "OPEN").
        2.  **Character Omission:** Exactly ONE missing character (e.g., "Invoce" vs "Invoice").
        3.  **Character Insertion:** Exactly ONE extra character (e.g., "Innvoice" vs "Invoice").
        4.  **Character Transposition:** Two adjacent characters swapped (e.g., "Ivnoice" vs "Invoice").
        5.  **Keyboard Proximity:** Exactly ONE character substituted by a QWERTY neighbor (e.g., "Invoicr" vs "Invoice").

3. **Synonymous Labels (Semantic Equivalence):**
   Labels that are syntactically different (often substantially) but share the same semantic meaning and represent the exact same business process step. 
   * **Detection Criteria (Ontology Rules):**
        1. **Linguistic & Domain Synonyms:** Different words representing the same concept within the process context (e.g., "Ship Item" vs "Dispatch Goods", "DrSeen" vs "Medical Assign").
        2. **Phrase Variation (Verb/Object Shift):** Labels sharing a core component (usually the Object) while using synonymous verbs or adjectives (e.g., "Create Invoice" vs "Generate Invoice", "Start instance" vs "Start process").
        3. **Grammatical Transformation:** Changing parts of speech (Noun ↔ Verb) or sentence structure while retaining the core meaning (e.g., "Give approval" vs "Approve", "Conduct analysis" vs "Analyze").
        4. **Containment & Refinement:** One label is a concise or verbose version of the other, often omitting non-essential adjectives, prepositions, or 'online/offline' qualifiers (e.g., "Receive signed contract" vs "Receive contract", "Register for course" vs "Register course").

### GLOBAL INSTRUCTION
- **Role:** Function as a logic engine. Do not assume all imperfections exist.
- **Priority:** The strict filtering logic in the **User Prompt** overrides general definitions here.
- **OUTPUT FORMAT:** Always return valid JSON as requested by the User Prompt.

"""

USER_PROMPT_SYNONYMOUS_STEP1 = f"""
### TASK: Filter Data for 'Synonymous Labels' Candidates

**OBJECTIVE:**
Analyze the provided **INPUT DATA(activity list)** and extract labels related to **Synonymous Labels (Semantic Equivalence)**.
You must **identify and collect** the following components for the output data:
1. All labels that form a **Synonym Group** (two or more labels representing the same process step despite syntactic differences).

**STRICT FILTERING LOGIC:**
1. **Identify Synonym Pairs:** Look for different words or phrases that share the same semantic meaning based on the System Prompt's criteria (Linguistic Synonyms, Phrase Variation, Grammatical Transformation, Containment).
2. **KEEP Related Labels:** If Label A is a semantic synonym of Label B, **KEEP BOTH A and B**. (Unlike Distorted/Polluted, typically all members of a synonym group are valid words, so keep the entire group).
3. **DISCARD Isolated Labels:** If a label is valid but has **NO** semantic synonyms in the provided list, **REMOVE IT**. (e.g., If 'Archive' exists but no synonyms like 'Store' or 'Save' exist, remove 'Archive').
4. **DISCARD Distorted/Polluted:** Remove labels that are purely 'Distorted Labels' (typos) or 'Polluted Labels' (IDs) if they are not part of a 'Synonymous Labels' pattern.

**EDGE CASE HANDLING:**
- If NO Synonymous pairs are found (i.e., all labels are unique/isolated, distorted, or polluted):
    - Return strictly `[]` (with "found": false).
- **Finding NOTHING is a valid result.** Do not force-fit vaguely similar words; strict semantic equivalence is required.

**INPUT DATA:**
{activity_list_json}

***OUTPUT FORMAT GUIDELINES (PERFORMANCE OPTIMIZED)***
Return a JSON Object with two keys:
1. "found": Boolean (true if synonymous labels exist, false otherwise).
2. "data": List of strings.

**Example (Found):**
{{ "found": true, "data": ["Create Invoice", "Generate Invoice", "Make Bill"] }}

**Example (Not Found - SPEED PRIORITY):**
{{ "found": false, "data": [] }}

**CONSTRAINT:**
- Determine the "found" value FIRST. If false, output `[]` for data immediately.
- Output ONLY the JSON.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_STEP1},
          {"role":"user","content": USER_PROMPT_SYNONYMOUS_STEP1}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
test_output = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
synonym_list_json = json.dumps(test_output['data'], indent=4, ensure_ascii=False)
for k, v in test_output.items():
    print(f"{k}:")
    if isinstance(v, list):  
        if v:  
            print(*v, sep="\n")
        else:  
            print(" (empty list)")
    else: 
        print(f" {v}")
    print() 

found:
 True

data:
Check for completeness
assess completeness
verify completeness
check complete
Perform checks
execute checks
check application
check no missing
Make decision
decide
take decision
send notification reject
notify reject
send notification accept
Notify accept
Deliver card
send card
deliver package
send package
New online application received
application received
get new application
review request received
Request info
require additional details
request more detail
request more information
information received
info received
obtain information



In [5]:
import pm4py

def get_synonym_context(df: pd.DataFrame,
                        case_col: str = 'case_id',
                        time_col: str = 'timestamp',
                        act_col: str = 'activity',
                        filter_list: set = None):
    df_pm4py = df[[case_col, time_col, act_col]].copy()
    df_pm4py.rename(columns={
        case_col: "case:concept:name",
        time_col: "time:timestamp",
        act_col: "concept:name"
    }, inplace=True)
    df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"], errors="coerce")
    dfg, start_activities, end_activities = pm4py.discover_dfg(df_pm4py)
    def get_activity_context(activity, dfg_dict):
        predecessors = {k[0]: v for k, v in dfg_dict.items() if k[1] == activity}
        successors = {k[1]: v for k, v in dfg_dict.items() if k[0] == activity}
        total_pred = sum(predecessors.values())
        total_succ = sum(successors.values())
        def format_to_list(dist_dict, total):
            if total == 0: return []
            items = [(k, v/total) for k, v in dist_dict.items() if (v/total) >= 0.05]
            items.sort(key=lambda x: x[1], reverse=True)
            return [k for k, v in items]
        return format_to_list(predecessors, total_pred), format_to_list(successors, total_succ)
    all_activities = sorted(df[act_col].unique())
    flow_data_list = []
    for act in all_activities:
        if filter_list is not None and act not in filter_list:
            continue
        pred, succ = get_activity_context(act, dfg)
        flow_data_list.append({
            'activity': act,
            'predecessors': pred, # 이제 리스트입니다 ['A', 'B']
            'successors': succ    # 이제 리스트입니다 ['C', 'D']
        })
        
    json_flow_context = json.dumps(flow_data_list, indent=2, ensure_ascii=False)
    return json_flow_context


target_activities_set = test_output['data']
synonym_context_json = get_synonym_context(
    df=df_new,          
    filter_list=target_activities_set 
)
print(synonym_context_json)

[
  {
    "activity": "Check for completeness",
    "predecessors": [
      "info received",
      "review request received",
      "information received",
      "obtain information",
      "revision received"
    ],
    "successors": [
      "New online application received",
      "Request info",
      "Perform checks",
      "application received",
      "get new application"
    ]
  },
  {
    "activity": "Deliver card",
    "predecessors": [
      "Notify accept",
      "send notification accept"
    ],
    "successors": [
      "EVENT 13 END"
    ]
  },
  {
    "activity": "Make decision",
    "predecessors": [
      "Perform checks",
      "check application",
      "execute checks"
    ],
    "successors": [
      "notify reject",
      "Notify accept",
      "send notification accept",
      "send notification reject"
    ]
  },
  {
    "activity": "New online application received",
    "predecessors": [
      "Check for completeness",
      "check no missing",
      "verify c

In [6]:
SYSTEM_PROMPT_SYNONYM_STEP2  = """
You are an expert Process Mining Analyst.
Your goal is to summarize lists of activity labels into a single, descriptive **Process Stage Name**.

### CORE TASK
You will be given an activity and its lists of **Predecessors** (incoming flow) and **Successors** (outgoing flow).
You must analyze the labels in each list and determine the **Common Business Phase** they represent.

### SUMMARIZATION LOGIC (ABSTRACTION)
1. **Identify the Core Action:** Look at the verbs and objects in the list.
2. **Ignore Noise:** Disregard synonyms, typos, and minor variations.
3. **Formulate a Summary:** Create a short, natural language phrase that encapsulates the collective meaning.

### EXAMPLES (Demonstration Only)
- **Input List:** `["Wrap package", "Box items", "Pack goods", "Containerize"]`
- **Output Summary:** "Packaging Phase"

- **Input List:** `["MRI Scan", "X-Ray taken", "Blood test results"]`
- **Output Summary:** "Medical Diagnosis Stage"

- **Input List:** `["Ticket Resolved", "Issue Fixed", "Close Ticket", "Problem Solved"]`
- **Output Summary:** "Ticket Resolution"

### GLOBAL INSTRUCTION
- **Input:** JSON object with `activity`, `predecessors` (list), and `successors` (list).
- **Output:** JSON object where `predecessors` and `successors` are converted to **Strings** (Summaries).
"""

USER_PROMPT_SYNONYM_STEP2 = f"""
### TASK: Summarize Contextual Flow Lists

**OBJECTIVE:**
Analyze the **INPUT DATA**. Replace the list of strings in `predecessors` and `successors` with a **Single Summarized String** describing that process stage.

**STRICT EXECUTION STEPS:**
1. **Iterate** through every activity in the input.
2. **Analyze Predecessors:**
   - Read the list of predecessor labels.
   - Abstract their common meaning into one short phrase (e.g., "Quality Check Phase").
   - **Replace** the list with this string.
3. **Analyze Successors:**
   - Read the list of successor labels.
   - Abstract their common meaning into one short phrase.
   - **Replace** the list with this string.

**INPUT DATA:**
{synonym_context_json}

***OUTPUT FORMAT GUIDELINES***
Return a JSON Object with a single key `"summarized_context"`.
The value must be a list of objects where `predecessors` and `successors` are **STRINGS**, not lists.

**Example Output (Mental Model):**
{{
  "summarized_context": [
    {{
      "activity": "Ship Item",
      "predecessors": "Packaging Phase",     // Was ["Box items", "Wrap package"...]
      "successors": "Delivery Initiation"    // Was ["Truck loaded", "Dispatch"...]
    }},
    {{
      "activity": "Handle Error",
      "predecessors": "System Failure",      // Was ["Crash", "Server Down"...]
      "successors": "Recovery Process"       // Was ["Reboot", "Restart"...]
    }}
  ]
}}

**Constraint:**
- Output **ONLY** the JSON object.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_SYNONYM_STEP2},
          {"role":"user","content": USER_PROMPT_SYNONYM_STEP2}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
test_output_synonym_step2 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
step2_out_json = json.dumps(test_output_synonym_step2["summarized_context"], indent=2, ensure_ascii=False)
print(step2_out_json)

[
  {
    "activity": "Check for completeness",
    "predecessors": "Application or additional information received",
    "successors": "Application registration and initial assessment"
  },
  {
    "activity": "Deliver card",
    "predecessors": "Acceptance notification to customer",
    "successors": "Process completion"
  },
  {
    "activity": "Make decision",
    "predecessors": "Detailed application checks completed",
    "successors": "Communicate acceptance or rejection decision"
  },
  {
    "activity": "New online application received",
    "predecessors": "Completeness verification of provided information",
    "successors": "Further checks or additional information request"
  },
  {
    "activity": "Notify accept",
    "predecessors": "Final approval decision made",
    "successors": "Card/package dispatch to customer"
  },
  {
    "activity": "Perform checks",
    "predecessors": "Application intake and completeness confirmation",
    "successors": "Decision-making on appl

In [7]:
SYSTEM_PROMPT_SYNONYM_STEP3 ="""
You are an expert Process Mining Analyst.
Your goal is to cluster activity labels into groups that represent the **Same Process Step**.

### INPUT DATA
You will receive objects with:
1. `activity`: The label.
2. `predecessors`: A summarized string (Input Context).
3. `successors`: A summarized string (Output Context).

### CLUSTERING LOGIC: FUZZY CONTEXT & SYNONYM BOOST
Compare pairs of activities (A and B). Decide if they are the same step based on two factors:

**FACTOR 1: CONTEXT SIMILARITY (The Base Rule)**
- Compare `predecessors_A` vs `predecessors_B` AND `successors_A` vs `successors_B`.
- **Do not look for exact string matches.**
- **Rule:** If the descriptions describe the **Same Business Phase** despite different wording, count it as a MATCH.

**FACTOR 2: LABEL SYNONYM BOOST (The Tie-Breaker)**
- **Rule:** If `activity_A` and `activity_B` are **Linguistic Synonyms** (e.g., "Verify" vs "Check"), you must be **MORE LENIENT** with context matching.
- **Logic:** "If labels imply the same action, allow minor deviations in context phrasing."

### FINAL DECISION MATRIX
1. **Contexts are Semantically Similar:** -> **GROUP**.
2. **Contexts have minor differences BUT Labels are Synonyms:** -> **GROUP** (Synonym Boost).
3. **Contexts are clearly different (Input or Output diverges):** -> **SEPARATE**.

### GLOBAL INSTRUCTION
- **Output:** A JSON object with a single key `"clusters"` containing a **List of Lists**.
- **Constraint:** Ensure Transitivity (A=B, B=C -> A=B=C).
"""

USER_PROMPT_SYNONYM_STEP3 = f"""
### TASK: Fuzzy Context Clustering with Synonym Boost

**OBJECTIVE:**
Group the activities in **INPUT DATA** that represent the same process step.
**Key Instruction:** Be flexible with context descriptions. Focus on the **Core Meaning**.

**STRICT EXECUTION STEPS:**

1. **Analyze Contexts:**
   - Read the natural language summaries.
   - Interpret different phrases describing the same stage as the **SAME** context (e.g., "Data Entry" ≈ "Inputting Data").

2. **Apply Clustering Logic (Use these Mental Models):**
   - **Case 1 (Strong Match):**
     - Context A: "User entering credentials"
     - Context B: "Inputting login details"
     - **Decision:** Contexts mean the same thing. -> **GROUP**.
   - **Case 2 (Synonym Boost):**
     - Labels: "Resolve Ticket" vs "Fix Issue" (Strong Synonyms).
     - Contexts: "Code review" vs "Peer review completed" (Slight wording diff).
     - **Decision:** Labels are synonyms, so ignore the slight context difference. -> **GROUP**.
   - **Case 3 (Mismatch):**
     - Labels: "Approve" vs "Reject".
     - Contexts: "Evaluation" vs "Evaluation". (Context match, but Labels opposite).
     - **Decision:** Clearly different outcome. -> **SEPARATE**.

3. **Apply Transitivity:**
   - Merge all overlapping pairs into final clusters.

**INPUT DATA (Summarized Context):**
{step2_out_json}

***OUTPUT FORMAT GUIDELINES***
Return a strict JSON Object with a single key `"clusters"`.

**Constraint:**
- Output **ONLY** the JSON object.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_SYNONYM_STEP3},
          {"role":"user","content": USER_PROMPT_SYNONYM_STEP3}] 
# USER_PROMPT_SYNONYMOUS_STEP1 , USER_PROMPT_POLLUTED_STEP1, USER_PROMPT_DISTORTED_STEP1
test_output_synonym_step3 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
step3_out = test_output_synonym_step3['clusters']

print(step3_out)

[['Check for completeness', 'assess completeness', 'check complete', 'check no missing', 'verify completeness'], ['New online application received', 'application received', 'get new application'], ['Perform checks', 'check application', 'execute checks'], ['Make decision', 'decide', 'take decision'], ['Notify accept', 'send notification accept'], ['notify reject', 'send notification reject'], ['Deliver card', 'deliver package', 'send card', 'send package'], ['Request info', 'request more detail', 'request more information', 'require additional details'], ['info received', 'information received', 'obtain information'], ['review request received']]


In [8]:
activity_counts = df_new['activity'].value_counts().to_dict()
synonym_dict = {}
for cluster in step3_out:
    if len(cluster) < 2:
        continue
    clean_label = max(cluster, key=lambda x: activity_counts.get(x, 0))
    variants = sorted([label for label in cluster if label != clean_label])
    synonym_dict[clean_label] = variants
print("\n--- [Process Abstraction Report] ---\n")
for clean, vars in sorted(synonym_dict.items()):
    print(f"Original: '{clean}' -> {vars})")


step4_out_json = json.dumps(synonym_dict, indent=4, ensure_ascii=False)




--- [Process Abstraction Report] ---

Original: 'Check for completeness' -> ['assess completeness', 'check complete', 'check no missing', 'verify completeness'])
Original: 'Deliver card' -> ['deliver package', 'send card', 'send package'])
Original: 'Make decision' -> ['decide', 'take decision'])
Original: 'New online application received' -> ['application received', 'get new application'])
Original: 'Notify accept' -> ['send notification accept'])
Original: 'Perform checks' -> ['check application', 'execute checks'])
Original: 'Request info' -> ['request more detail', 'request more information', 'require additional details'])
Original: 'info received' -> ['information received', 'obtain information'])
Original: 'notify reject' -> ['send notification reject'])


In [24]:
def evaluate_synonym_detection(df_new, target_case_ids, synonym_dict):
    filtered_df = df_new[df_new['case_id'].astype(str).isin(target_case_ids)].copy()
    
    all_synonym_variants = set()
    for variants in synonym_dict.values():
        all_synonym_variants.update(variants)
        
    filtered_df['is_detected'] = filtered_df['activity'].isin(all_synonym_variants)
    filtered_df['is_correct'] = filtered_df['is_detected'] & filtered_df['label'].notna()
    
    return filtered_df
    
target_cases = cases_json[:20]  
target_case_ids = set()       
for batch in target_cases:
    for event in batch:
        c_id = str(event.get('case_id'))
        target_case_ids.add(c_id)

result_df = evaluate_synonym_detection(df_new, target_case_ids, synonym_dict)

detected_rows = result_df[result_df['is_detected']]
total_detected = len(detected_rows)
correct_detected = detected_rows['is_correct'].sum()

print("=== Evaluation Results ===")
print(f"1. Total Detected (Predicted Positives): {total_detected}")
print(f"2. Correct Matches (True Positives): {correct_detected}")

if total_detected > 0:
    precision = (correct_detected / total_detected) * 100
    print(f"3. Precision: {precision:.2f}%")
else:
    print("3. Precision: N/A (No detections)")

print("\n[Detailed View: Top 10 Detected Events]")
print(detected_rows[['case_id', 'activity', 'label', 'is_correct']].head(10))

=== Evaluation Results ===
1. Total Detected (Predicted Positives): 635
2. Correct Matches (True Positives): 635
3. Precision: 100.00%

[Detailed View: Top 10 Detected Events]
   case_id                    activity                       label  is_correct
2       15           check application  synonymous Label(Activity)        True
4       15    send notification reject  synonymous Label(Activity)        True
7       19            check no missing  synonymous Label(Activity)        True
8       19        application received  synonymous Label(Activity)        True
10      19        information received  synonymous Label(Activity)        True
11      19         assess completeness  synonymous Label(Activity)        True
14      19    send notification reject  synonymous Label(Activity)        True
16      19         verify completeness  synonymous Label(Activity)        True
27      22  require additional details  synonymous Label(Activity)        True
28      22        information rece

In [19]:
SYSTEM_PROMPT_SYNONYM_STEP5_STRICT = """
You are a strict Data Filtering Engine.
Your task is to extract event_ids based ONLY on exact string matching against a Reference Mapping.

### STRICT MATCHING RULES
1. **Exact Match Only:** The `activity` string in the log must matches a string in the Synonym List **character-for-character**.
    * "Check" != "check" (Case sensitive)
    * "Check " != "Check" (Whitespace sensitive)
    * "Checking" != "Check" (No partial matches or stemming)
2. **Key Exclusion:** If the `activity` matches the **Key** (Standard Label) of the mapping, IGNORE it. It is already clean.
3. **No Inference:** Do NOT imply synonyms. If it's not explicitly in the list, ignore it.

### OUTPUT FORMAT
Return strictly a valid JSON object:
{ "event_id": ["id_1", "id_2"] }
"""

def get_synonym_strict_prompt(synonym_mapping, event_log_chunk):
    event_log_chunk = json.dumps(event_log_chunk, indent=2, ensure_ascii=False)
    return f"""
### TASK: Strict Log Filtering

**1. REFERENCE MAPPING:**
{synonym_mapping}

**2. TARGET EVENT LOG:**
{event_log_chunk}

***INSTRUCTIONS***
Scan the log. For each event:
1. Get the `activity` string.
2. Check if this **exact string** appears in any **Value List** within the Reference Mapping.
3. If yes, add `event_id` to output.
4. **DO NOT** use semantic similarity. Look for identical strings only.

***OUTPUT***
JSON Object with matching event_ids.
"""
target_cases = cases_json[:20]  
target_case_ids = set()       
for batch in target_cases:
    for event in batch:
        c_id = str(event.get('case_id'))
        target_case_ids.add(c_id)
all_predicted_ids = set() 
for case in target_cases:
    try:
        USER_PROMPT = get_synonym_user_prompt_step5(step4_out_json, case)
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT_SYNONYM_STEP5},
            {"role": "user", "content": USER_PROMPT}
        ]
        test_output = llm_gen(model_version=model_, model_instance=llm, prompt=prompt)
        if isinstance(test_output, dict):
            current_ids = test_output.get('event_id', [])
            all_predicted_ids.update(str(x) for x in current_ids)
            
    except Exception as e:
        print(f"Error processing a case: {e}")
        continue
target_df = df_new[df_new['case_id'].astype(str).isin(target_case_ids)]
ground_truth_df = target_df[target_df['label'].notna()]
actual_ids = set(ground_truth_df['event_id'].astype(str))
intersection = all_predicted_ids.intersection(actual_ids)
tp = len(intersection)
n_actual = len(actual_ids) 
n_pred = len(all_predicted_ids) 
recall = (tp / n_actual * 100) if n_actual > 0 else 0.0
precision = (tp / n_pred * 100) if n_pred > 0 else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print("\n" + "=" * 40)
print(f"   BATCH EVALUATION ({len(target_case_ids)} CASES)   ")
print("=" * 40)
print(f"1. Total Ground Truth (Actual) : {n_actual}")
print(f"2. Total Predictions (Model)   : {n_pred}")
print(f"3. True Positives (Matches)    : {tp}")
print("-" * 40)
print(f"▶ Recall    : {recall:.2f}%")
print(f"▶ Precision : {precision:.2f}%")
print(f"▶ F1 Score  : {f1_score:.2f}")
print("=" * 40)



   BATCH EVALUATION (200 CASES)   
1. Total Ground Truth (Actual) : 651
2. Total Predictions (Model)   : 659
3. True Positives (Matches)    : 588
----------------------------------------
▶ Recall    : 90.32%
▶ Precision : 89.23%
▶ F1 Score  : 89.77


In [21]:
SYSTEM_PROMPT_SYNONYM_STEP5_BETA = """
You are an expert Data Quality Analyst and Process Mining Specialist.
Your task is to identify specific events in an Event Log that contain **"Synonym Variants"** (non-standard labels).
You must detect these variants using two methods: (1) A provided Reference Mapping, and (2) On-the-fly Semantic & Contextual Discovery.

### 1. KNOWLEDGE BASE: SYNONYMOUS LABELS
Use these criteria to determine if two different labels are synonyms:
1. **Linguistic & Domain Synonyms:** Different words representing the same concept (e.g., "Ship Item" vs "Dispatch Goods").
2. **Phrase Variation:** Verb/Object shifts sharing a core component (e.g., "Create Invoice" vs "Generate Invoice").
3. **Grammatical Transformation:** Part of speech changes (e.g., "Give approval" vs "Approve").
4. **Containment & Refinement:** Concise vs Verbose versions (e.g., "Receive signed contract" vs "Receive contract").

### 2. DETECTION LOGIC

#### METHOD A: Reference Mapping Filter
Check every event's `activity` against the provided **Reference Mapping**.
* **Target:** If the activity exists in the **Values (Lists)** of the mapping.
* **Action:** Collect the `event_id`.
* **Exclusion:** Do NOT collect if the activity matches the **Key** (Standard Label).

#### METHOD B: Unmapped Contextual Discovery (New Pairs)
Scan the event log for **NEW** synonym pairs that are **NOT** in the Reference Mapping (neither Key nor Value).
* **Condition 1 (Semantic):** Two different labels appear to be synonyms based on the 'Knowledge Base' above.
* **Condition 2 (Structural/Flow):** These labels appear in different cases but occupy the **same process step** (e.g., they share similar predecessors/successors or timestamps relative to the case start).
* **Action:** If Activity X (Case A) and Activity Y (Case B) meet these conditions, collect `event_id` for **BOTH** events.

### OUTPUT FORMAT
Return strictly a valid JSON object containing a single key "event_id" with a list of strings.
No markdown, no explanations.

**Example Structure:**
{
  "event_id": ["id_123", "id_999", "id_new_discovery_1", "id_new_discovery_2"]
}
"""

def get_synonym_user_prompt_step5_beta(synonym_mapping, event_log_chunk):
    event_log_chunk = json.dumps(event_log_chunk, indent=2, ensure_ascii=False)
    return f"""
### TASK: Extract Event IDs (Known Variants + New Contextual Synonyms)

**OBJECTIVE:**
Scan the **Target Event Log**. Identify `event_id`s for activities that are either **Known Variants** OR **New Contextual Synonyms**.

**1. REFERENCE MAPPING (Standard -> [Variants]):**
{synonym_mapping}

**2. TARGET EVENT LOG (Multi-case Stream):**
{event_log_chunk}

***INSTRUCTIONS***

**Step 1: Check Known Mapping**
- Is the `activity` found in the **Values** of the Reference Mapping?
- If YES -> Keep `event_id`.

**Step 2: Discover New Synonyms (Contextual Analysis)**
- Look for activity pairs in the log that are **NOT** in the Reference Mapping.
- Apply the **Synonym Definition** (Linguistic, Phrase Variation, etc.).
- **CRITICAL:** Verify **Process Flow Context**.
    - *Example:* If Case 1 has `A -> B -> C` and Case 2 has `A -> B' -> C`, and `B` & `B'` are semantically similar, they are synonyms.
- If YES (Semantic + Context match) -> Keep `event_id` for **BOTH** events.

***OUTPUT***
Return ONLY the JSON object with the combined list of matching event IDs.
"""
    
target_cases = cases_json[:20]  
target_case_ids = set()       
for batch in target_cases:
    for event in batch:
        c_id = str(event.get('case_id'))
        target_case_ids.add(c_id)
all_predicted_ids = set() 
for case in target_cases:
    try:
        USER_PROMPT = get_synonym_user_prompt_step5_beta(step4_out_json, case)
        prompt = [
            {"role": "system", "content": SYSTEM_PROMPT_SYNONYM_STEP5_BETA},
            {"role": "user", "content": USER_PROMPT}
        ]
        test_output = llm_gen(model_version=model_, model_instance=llm, prompt=prompt)
        if isinstance(test_output, dict):
            current_ids = test_output.get('event_id', [])
            all_predicted_ids.update(str(x) for x in current_ids)
            
    except Exception as e:
        print(f"Error processing a case: {e}")
        continue
target_df = df_new[df_new['case_id'].astype(str).isin(target_case_ids)]
ground_truth_df = target_df[target_df['label'].notna()]
actual_ids = set(ground_truth_df['event_id'].astype(str))
intersection = all_predicted_ids.intersection(actual_ids)
tp = len(intersection)
n_actual = len(actual_ids) 
n_pred = len(all_predicted_ids) 
recall = (tp / n_actual * 100) if n_actual > 0 else 0.0
precision = (tp / n_pred * 100) if n_pred > 0 else 0.0
f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print("\n" + "=" * 40)
print(f"   BATCH EVALUATION ({len(target_cases)} CASES)   ")
print("=" * 40)
print(f"1. Total Ground Truth (Actual) : {n_actual}")
print(f"2. Total Predictions (Model)   : {n_pred}")
print(f"3. True Positives (Matches)    : {tp}")
print("-" * 40)
print(f"▶ Recall    : {recall:.2f}%")
print(f"▶ Precision : {precision:.2f}%")
print(f"▶ F1 Score  : {f1_score:.2f}")
print("=" * 40)



   BATCH EVALUATION (20 CASES)   
1. Total Ground Truth (Actual) : 651
2. Total Predictions (Model)   : 751
3. True Positives (Matches)    : 564
----------------------------------------
▶ Recall    : 86.64%
▶ Precision : 75.10%
▶ F1 Score  : 80.46
